In [ ]:
from typing import TypedDict

from dotenv import  load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph,START,END
from loguru import logger
from IPython.display import  display
load_dotenv(override=True)


model=ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

class OverAllState(TypedDict):
    topic:str
    poem:str
    joke:str
    final_output:str

class InputState(TypedDict):
    topic:str

class OutputState(TypedDict):
    final_output:str

def node_poem(state:InputState)->OverAllState:
    logger.info("node_poem正在执行")
    topic=state["topic"]
    poem=model.invoke([HumanMessage(f"写一首关于{topic}主题的七言绝句")]).content
    return {
        "poem":poem
    }


def node_joke(state:InputState)->OverAllState:
    logger.info("node_joke正在执行")
    topic=state["topic"]
    joke=model.invoke([HumanMessage(f"写一个关于{topic}主题的笑话")]).content
    return {
        "joke":joke
    }

def Output_node(state:OverAllState)->OutputState:
    logger.info("Output_node正在执行")
    topic=state["topic"]
    poem=state["poem"]
    joke=state["joke"]
    final_output=f"关于{topic}主题的七言绝句:{poem}，\n笑话:{joke}\n"
    return {
        "final_output":final_output
    }

builder=StateGraph(state_schema=OverAllState,input_schema=InputState,output_schema=OutputState)
builder.add_node("node_poem",node_poem)
builder.add_node("node_joke",node_joke)
builder.add_node("Output_node",Output_node)
builder.add_edge(START,"node_poem")
builder.add_edge(START,"node_joke")
builder.add_edge("node_poem","Output_node")
builder.add_edge("node_joke","Output_node")
builder.add_edge("Output_node",END)

checkpointer=InMemorySaver()
config={
    "configurable":{
        "thread_id":"123"
    }
}
graph=builder.compile(checkpointer=checkpointer)
res=graph.invoke({"topic":"莲花"},config=config)
print(res)
display(graph)

In [ ]:
history_checkpoints = list(graph.get_state_history(config=config))
print(history_checkpoints)

In [ ]:
latest_history_checkpoint = graph.get_state(config=config)
print(latest_history_checkpoint)

In [ ]:
target_config = {
    "configurable": {
        "thread_id": "123",
        "checkpoint_id": history_checkpoints[0].config["configurable"]["checkpoint_id"]
    }
}

snapshot = graph.get_state(config=target_config)
print(snapshot)

In [ ]:
print('=' * 30, '-> 根据ID查看特定检查点 <-', '=' * 30)
checkpointe_id = history_checkpoints[-2].config["configurable"]["checkpoint_id"]
print(f"checkpointe_id: {checkpointe_id}")

target_config = {
    "configurable": {
        "thread_id": config["configurable"]["thread_id"],
        "checkpoint_id": checkpointe_id
    }
}

specific_history_checkpoint = graph.get_state(config=target_config)
print(specific_history_checkpoint)